<a href="https://colab.research.google.com/github/rajidush/RP-J26-DS-316/blob/c3%2Fmodel-tuning/component3_socratic_educator/notebooks/train_inquire_lora_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train the INQUIRE LoRA adapter (Component 3)

Fine-tunes `google/gemma-3-1b-it` with the **fixed** LoRA config in `inquire_training.py`
(r=16, alpha=32, attention + MLP projections, LR 1e-4, 3 epochs, completion-only loss),
evaluating and saving every epoch and keeping the checkpoint with the lowest validation loss.
There is no hyperparameter search.

**Before you start**
- Runtime → Change runtime type → **T4 GPU**.
- Colab secret `HF_TOKEN` (🔑 in the sidebar) for a Hugging Face account that has accepted the Gemma licence.
- From the repo: `component3_socratic_educator/inquire_training.py` and the four files in
  the `build` folder named by `DATASET_FOLDER` below, e.g. `component3_socratic_educator/evidence/inquire_dataset_2026-10-10/` (`manifest.json`, `train.jsonl`, `validation.jsonl`, `test.jsonl`).

**Outputs** go to Google Drive only (`MyDrive/c3_inquire_training/<run id>/`): checkpoints, the adapter and `run_record.json`.
Only `run_record.json` comes back to the repo, as evidence. The adapter never does.

**Comparison (ticket 06)** is the last section. It loads the adapter from Drive, so after the setup cells
(install, Drive, upload, manifest check, GPU) you can skip the training cells and run it on its own.

In [1]:
!pip -q install "transformers==4.56.2" "trl==0.23.1" "peft==0.17.1" "accelerate==1.10.1" "datasets==4.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.6/564.6 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This be

In [2]:
import time
from google.colab import drive
drive.mount("/content/drive")

RUN_ID = time.strftime("%Y-%m-%d_%H%M", time.gmtime())
RUN_DIR = f"/content/drive/MyDrive/c3_inquire_training/{RUN_ID}"
ADAPTER_DIR = f"{RUN_DIR}/adapter"
DATA_DIR = "/content/inquire_dataset"
DATASET_FOLDER = "inquire_dataset_2026-10-10"   # the repo `build` folder you upload from (recorded in the run record)
print("run directory:", RUN_DIR)

Mounted at /content/drive
run directory: /content/drive/MyDrive/c3_inquire_training/2026-10-10_1350


Upload `inquire_training.py`, `manifest.json`, `train.jsonl`, `validation.jsonl` and `test.jsonl` (select all five at once).

In [3]:
import os
import re
from google.colab import files

EXPECTED = ("inquire_training.py", "manifest.json", "train.jsonl", "validation.jsonl", "test.jsonl")
uploaded = files.upload()
os.makedirs(DATA_DIR, exist_ok=True)
for name in uploaded:
    base = re.sub(r" \(\d+\)(?=\.)", "", name)  # Colab renames re-uploads to 'file (1).jsonl'
    if base not in EXPECTED:
        print("ignored:", name)
        continue
    os.replace(name, base if base == "inquire_training.py" else f"{DATA_DIR}/{base}")
missing = [n for n in EXPECTED if not os.path.exists(n if n.endswith(".py") else f"{DATA_DIR}/{n}")]
assert not missing, f"still missing: {missing}"
print("uploaded:", ", ".join(EXPECTED))

Saving inquire_training.py to inquire_training.py
Saving manifest.json to manifest.json
Saving test.jsonl to test.jsonl
Saving train.jsonl to train.jsonl
Saving validation.jsonl to validation.jsonl
uploaded: inquire_training.py, manifest.json, train.jsonl, validation.jsonl, test.jsonl


Stop here unless the dataset is exactly the `build` output its manifest describes.

In [4]:
import inquire_training as tr

manifest = tr.verify_dataset(DATA_DIR)   # raises DatasetError on a missing or mismatched file
MANIFEST_SHA256 = tr.manifest_sha256(DATA_DIR)
for split in ("train", "validation", "test"):
    c = manifest["counts"][split]
    print(f"{split:<10} {c['rows']:>3} rows, {c['scenarios']:>2} scenarios")
print("manifest sha256:", MANIFEST_SHA256)

train      104 rows, 41 scenarios
validation  25 rows,  5 scenarios
test        50 rows, 10 scenarios
manifest sha256: 444a5aae4fc83ec74cc0d1dc4e757a976b7725d00c1eb5cb004fcde701f46eba


In [5]:
import torch
from google.colab import userdata
from huggingface_hub import login

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
GPU = torch.cuda.get_device_name(0)
login(token=userdata.get("HF_TOKEN"))
print("GPU:", GPU, "| precision:", tr.PRECISION)

GPU: Tesla T4 | precision: fp32


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

set_seed(tr.SEED)
tokenizer = AutoTokenizer.from_pretrained(tr.BASE_MODEL)
# fp32 (see tr.PRECISION_REASON); eager attention is the recommended setting for training Gemma 3.
model = AutoModelForCausalLM.from_pretrained(tr.BASE_MODEL, dtype=torch.float32, attn_implementation="eager")
MODEL_REVISION = getattr(model.config, "_commit_hash", None)
print("model revision:", MODEL_REVISION)

In [ ]:
from datasets import load_dataset

data = load_dataset("json", data_files={"train": f"{DATA_DIR}/train.jsonl",
                                        "validation": f"{DATA_DIR}/validation.jsonl"})
data = data.select_columns(["prompt", "completion"])   # metadata stays in the files, not in training
lengths = [len(tokenizer.apply_chat_template(r["prompt"] + r["completion"], tokenize=True))
           for split in data.values() for r in split]
MAX_TOKENS = max(lengths)
assert MAX_TOKENS <= tr.TRAINER_CONFIG["max_length"], f"{MAX_TOKENS} tokens would be truncated"
print(data)
print("longest prompt + completion:", MAX_TOKENS, "tokens")

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

args = SFTConfig(output_dir=f"{RUN_DIR}/checkpoints", **tr.TRAINER_CONFIG)
trainer = SFTTrainer(model=model, args=args, train_dataset=data["train"], eval_dataset=data["validation"],
                     processing_class=tokenizer, peft_config=LoraConfig(**tr.LORA_CONFIG))
assert "completion_mask" in trainer.train_dataset.column_names, "loss would not be completion-only"
TRAINABLE = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
trainer.model.print_trainable_parameters()

In [ ]:
import math

STARTED_AT = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
start = time.time()
trainer.train()
RUNTIME = round(time.time() - start, 1)

LOSSES = tr.epoch_losses(trainer.state.log_history)
for row in LOSSES:
    print(row)
bad = [e for e in LOSSES if not (isinstance(e["eval_loss"], float) and math.isfinite(e["eval_loss"]))]
assert not bad, f"missing or non-finite validation loss: {bad}"
print("kept:", trainer.state.best_model_checkpoint, "eval_loss", trainer.state.best_metric, "| runtime", RUNTIME, "s")

`load_best_model_at_end` has already put the lowest-validation-loss checkpoint back into `trainer.model`; save that adapter.

In [ ]:
trainer.save_model(ADAPTER_DIR)
print("adapter saved to Drive:", ADAPTER_DIR)

In [ ]:
import importlib.metadata as md
import json
import platform

def version(package):
    try:
        return md.version(package)
    except md.PackageNotFoundError:
        return None

versions = {p: version(p) for p in ("torch", "transformers", "trl", "peft", "accelerate", "datasets")}
versions["python"] = platform.python_version()

record = tr.run_record(
    manifest_sha256=MANIFEST_SHA256, dataset_folder=DATASET_FOLDER, versions=versions, gpu=GPU,
    runtime_seconds=RUNTIME, log_history=trainer.state.log_history,
    best_checkpoint=trainer.state.best_model_checkpoint, best_metric=trainer.state.best_metric,
    adapter_location=ADAPTER_DIR,
    extra={"run_id": RUN_ID, "started_at": STARTED_AT, "model_revision": MODEL_REVISION, "cuda": torch.version.cuda,
           "train_rows": len(data["train"]), "validation_rows": len(data["validation"]),
           "max_prompt_completion_tokens": MAX_TOKENS, "trainable_parameters": TRAINABLE,
           "pinned_packages": list(tr.PINNED_PACKAGES), "trainer_args_full": json.loads(args.to_json_string())})
tr.write_run_record(f"{RUN_DIR}/run_record.json", record)
files.download(f"{RUN_DIR}/run_record.json")

**After the run:** save the downloaded `run_record.json` as
`component3_socratic_educator/evidence/inquire_training_<date>/run_record.json` in the repo.
The adapter stays on Drive (`ADAPTER_DIR`); ticket 06 loads it from there for the comparison.

## Comparison: control vs adapted (ticket 06)

Both arms load the same base checkpoint at the same precision (fp32, eager attention). The **control** arm is the
base checkpoint alone. The **adapted** arm is a fresh copy of the base checkpoint with the LoRA adapter from Drive merged in
(`merge_and_unload`, in fp32), so both arms run the same architecture and latency isn't skewed by the PEFT wrapper.
Each arm answers every prompt in `test.jsonl` once, using the same chat template and the same settings
(`tr.GENERATION_CONFIG`: greedy, temperature 0, fixed `max_new_tokens`). Responses are stripped of whitespace
only and never repaired.

Raw outputs go to Drive next to the adapter (`comparison_<id>/raw_outputs_<arm>.jsonl`) and are then downloaded.
**Responses are not printed**, so you stay blind until you score the sheet.

Set `ADAPTER_SOURCE` to `adapter_location` from the run record you are comparing.

In [6]:
import gc
import json
import os
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

ADAPTER_SOURCE = "/content/drive/MyDrive/c3_inquire_training/2026-10-10_1249/adapter"   # run_record.json: adapter_location
assert os.path.isfile(f"{ADAPTER_SOURCE}/adapter_config.json"), f"no adapter at {ADAPTER_SOURCE}: check the run id"
COMPARISON_ID = time.strftime("%Y-%m-%d_%H%M", time.gmtime())
COMPARISON_DIR = f"{os.path.dirname(ADAPTER_SOURCE)}/comparison_{COMPARISON_ID}"
os.makedirs(COMPARISON_DIR)

for name in ("trainer", "model"):   # free the training run's model if this session trained one
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()

with open(f"{DATA_DIR}/test.jsonl", encoding="utf-8") as f:
    test_rows = [json.loads(line) for line in f]
tokenizer = AutoTokenizer.from_pretrained(tr.BASE_MODEL)
print(len(test_rows), "test prompts | outputs to", COMPARISON_DIR)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

50 test prompts | outputs to /content/drive/MyDrive/c3_inquire_training/2026-10-10_1249/comparison_2026-10-10_1355


In [7]:
from peft import PeftModel

def load_base():
    # Identical for both arms: the same checkpoint, fp32 (tr.PRECISION) and eager attention as in training.
    return AutoModelForCausalLM.from_pretrained(tr.BASE_MODEL, dtype=torch.float32, attn_implementation="eager").to("cuda")

def generator(m):
    @torch.inference_mode()
    def generate(messages):
        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt",
                                               return_dict=True).to(m.device)
        new = m.generate(**inputs, **tr.GENERATION_CONFIG)[0, inputs["input_ids"].shape[1]:]
        return tokenizer.decode(new, skip_special_tokens=True), int(new.shape[0])
    generate(test_rows[0]["prompt"])   # warm-up, not recorded, so the first timed call doesn't pay for CUDA setup
    return generate

def save_arm(arm, records):
    path = f"{COMPARISON_DIR}/raw_outputs_{arm}.jsonl"
    tr.write_raw_outputs(path, records)
    cut = sum(r["hit_max_new_tokens"] for r in records)
    mean = sum(r["latency_seconds"] for r in records) / len(records)
    print(f"{arm}: {len(records)} responses, mean latency {mean:.2f} s, {cut} cut off at max_new_tokens -> {path}")

In [8]:
import importlib.metadata as md
import platform

def version(package):
    try:
        return md.version(package)
    except md.PackageNotFoundError:
        return None

set_seed(tr.SEED)
control = load_base().eval()
MODEL_REVISION = getattr(control.config, "_commit_hash", None)
versions = {p: version(p) for p in ("torch", "transformers", "peft", "accelerate")}
versions["python"] = platform.python_version()
SETTINGS = tr.generation_settings(manifest_sha256=MANIFEST_SHA256, model_revision=MODEL_REVISION,
                                  versions=versions, gpu=GPU)

save_arm("control", tr.generate_arm(test_rows, "control", generator(control), settings=SETTINGS))
del control
gc.collect()
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


control: 50 responses, mean latency 1.44 s, 0 cut off at max_new_tokens -> /content/drive/MyDrive/c3_inquire_training/2026-10-10_1249/comparison_2026-10-10_1355/raw_outputs_control.jsonl


In [9]:
set_seed(tr.SEED)
adapted = PeftModel.from_pretrained(load_base(), ADAPTER_SOURCE).merge_and_unload().eval()
assert getattr(adapted.config, "_commit_hash", None) == MODEL_REVISION, "base checkpoint changed"

save_arm("adapted", tr.generate_arm(test_rows, "adapted", generator(adapted), settings=SETTINGS, adapter=ADAPTER_SOURCE))
del adapted
gc.collect()
torch.cuda.empty_cache()

adapted: 50 responses, mean latency 1.42 s, 0 cut off at max_new_tokens -> /content/drive/MyDrive/c3_inquire_training/2026-10-10_1249/comparison_2026-10-10_1355/raw_outputs_adapted.jsonl


In [10]:
from google.colab import files

for arm in tr.ARMS:
    files.download(f"{COMPARISON_DIR}/raw_outputs_{arm}.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**After the comparison:** save both downloaded files in
`component3_socratic_educator/evidence/inquire_comparison_<date>/` in the repo, then build the blind sheet from `component3_socratic_educator/`:

```
python inquire_finetune.py sheet --test evidence/inquire_dataset_2026-10-10/test.jsonl --outputs evidence/inquire_comparison_<date>/raw_outputs_control.jsonl evidence/inquire_comparison_<date>/raw_outputs_adapted.jsonl --out evidence/inquire_comparison_<date>
```

Don't open the raw-output files or `key.json` until every row of the sheet is scored.